In [ ]:

import pandas as pd

# Load the training and test datasets
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\DSEval\\datasets\\05_patient_profile\\train.csv'
test_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\DSEval\\datasets\\05_patient_profile\\test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# Display the first few rows of the training data to understand its structure
train_df.head()


Displayed on the user's machine.

In [ ]:

# One-hot encode the categorical features, excluding the 'Disease' column
categorical_features = ['Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Gender', 'Blood Pressure', 'Cholesterol Level']

# Perform one-hot encoding
train_df_encoded = pd.get_dummies(train_df, columns=categorical_features, drop_first=True)
test_df_encoded = pd.get_dummies(test_df, columns=categorical_features, drop_first=True)

# Display the first few rows of the encoded training data to verify
train_df_encoded.head()


Displayed on the user's machine.

In [ ]:


from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Extract the target variable and the features
X_train = train_df_encoded.drop(['Disease', 'Outcome Variable'], axis=1)
y_train = train_df_encoded['Outcome Variable'].map({'Positive': 1, 'Negative': 0})

# Initialize the logistic regression model
logreg = LogisticRegression(max_iter=1000)

# Perform Recursive Feature Elimination to select the top 5 features
rfe = RFE(estimator=logreg, n_features_to_select=5, step=1)
rfe.fit(X_train, y_train)

# Get the selected features
selected_features = X_train.columns[rfe.support_]
selected_features



Index(['Fever_Yes', 'Fatigue_Yes', 'Gender_Male', 'Cholesterol Level_Low',
       'Cholesterol Level_Normal'],
      dtype='object')

In [ ]:


# Build Model 1: Using all features
logreg_all = LogisticRegression(max_iter=1000)
logreg_all.fit(X_train, y_train)

# Build Model 2: Using only the selected features
X_train_selected = X_train[selected_features]
logreg_selected = LogisticRegression(max_iter=1000)
logreg_selected.fit(X_train_selected, y_train)

# Prepare the test data for evaluation
X_test = test_df_encoded.drop(['Disease', 'Outcome Variable'], axis=1)
y_test = test_df_encoded['Outcome Variable'].map({'Positive': 1, 'Negative': 0})

# Select the same features for the test set as were selected for the training set
X_test_selected = X_test[selected_features]

# Make predictions on the test set for both models
y_pred_all = logreg_all.predict(X_test)
y_pred_selected = logreg_selected.predict(X_test_selected)

# Evaluate the performance of both models
from sklearn.metrics import accuracy_score, classification_report

accuracy_all = accuracy_score(y_test, y_pred_all)
accuracy_selected = accuracy_score(y_test, y_pred_selected)

print("Model 1 (All Features) Accuracy:", accuracy_all)
print("Model 2 (Selected Features) Accuracy:", accuracy_selected)

# Print classification reports for more detailed performance metrics
print("\nModel 1 (All Features) Classification Report:\n", classification_report(y_test, y_pred_all))
print("\nModel 2 (Selected Features) Classification Report:\n", classification_report(y_test, y_pred_selected))



Model 1 (All Features) Accuracy: 0.5714285714285714
Model 2 (Selected Features) Accuracy: 0.5714285714285714

Model 1 (All Features) Classification Report:
               precision    recall  f1-score   support

           0       0.50      0.53      0.52        30
           1       0.63      0.60      0.62        40

    accuracy                           0.57        70
   macro avg       0.57      0.57      0.57        70
weighted avg       0.58      0.57      0.57        70


Model 2 (Selected Features) Classification Report:
               precision    recall  f1-score   support

           0       0.50      0.53      0.52        30
           1       0.63      0.60      0.62        40

    accuracy                           0.57        70
   macro avg       0.57      0.57      0.57        70
weighted avg       0.58      0.57      0.57        70

